In [ ]:
import pandas as pd
import numpy as np

train_woe = pd.read_csv("../data/processed/train_woe.csv")
validation_woe = pd.read_csv("../data/processed/validation_woe.csv")
test_woe = pd.read_csv("../data/processed/test_woe.csv")

In [ ]:
import statsmodels.api as sm

final_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "MonthlyIncome",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
    "NumberOfTimes90DaysLate"
]

In [ ]:
# final training of the model

X_train = train_woe[final_variables]
y_train = train_woe["SeriousDlqin2yrs"]

X_train = sm.add_constant(X_train)

model = sm.Logit(
    y_train,
    X_train
).fit()

print(model.summary())

## Probability Prediction

In [ ]:
train_woe["pd"] = model.predict(X_train)

In [ ]:
X_validation = validation_woe[final_variables]

X_validation = sm.add_constant(
    X_validation
)

validation_woe["pd"] = model.predict(
    X_validation
)

In [ ]:
X_test = test_woe[final_variables]

X_test = sm.add_constant(
    X_test
)

test_woe["pd"] = model.predict(
    X_test
)

## AUROC

In [ ]:
from sklearn.metrics import roc_auc_score

train_auc = roc_auc_score(
    y_train,
    train_woe["pd"]
)

print(train_auc)

In [ ]:
validation_auc = roc_auc_score(
    validation_woe["SeriousDlqin2yrs"],
    validation_woe["pd"]
)

print(validation_auc)

In [ ]:
test_auc = roc_auc_score(
    test_woe["SeriousDlqin2yrs"],
    test_woe["pd"]
)

print(test_auc)

## GINI

In [ ]:
train_gini = 2 * train_auc - 1
validation_gini = 2 * validation_auc - 1
test_gini = 2 * test_auc - 1

print(train_gini)
print(validation_gini)
print(test_gini)

## ROC Curve

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

In [ ]:
fpr, tpr, thresholds = roc_curve(
    validation_woe["SeriousDlqin2yrs"],
    validation_woe["pd"]
)

plt.figure(figsize=(8,6))

plt.plot(fpr, tpr)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.show()

## KS Statistic

In [ ]:
ks = max(
    tpr - fpr
)

print(ks)

## Overfitting Check

In [ ]:
performance = pd.DataFrame({
    "dataset": [
        "Train",
        "Validation",
        "Test"
    ],
    "AUROC": [
        train_auc,
        validation_auc,
        test_auc
    ],
    "GINI": [
        train_gini,
        validation_gini,
        test_gini
    ]
})

performance

### Model Performance Summary

The final scorecard model demonstrated strong discriminatory power across all samples.

Performance metrics remained highly stable between the training, validation and test datasets, indicating no evidence of overfitting.

Results:

- Train AUROC: 0.830
- Validation AUROC: 0.828
- Test AUROC: 0.828

- Train GINI: 65.9%
- Validation GINI: 65.6%
- Test GINI: 65.5%

- Validation KS: 51.5%

The minimal performance degradation between datasets confirms that the model generalizes well to unseen observations.

## Confusion Matrix

In [ ]:
cutoff = 0.10

In [ ]:
validation_woe["prediction"] = (
    validation_woe["pd"] >= cutoff
).astype(int)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    validation_woe["SeriousDlqin2yrs"],
    validation_woe["prediction"]
)

cm

In [ ]:
tn, fp, fn, tp = cm.ravel()

print("TN =", tn)
print("FP =", fp)
print("FN =", fn)
print("TP =", tp)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
accuracy_score(
    validation_woe["SeriousDlqin2yrs"],
    validation_woe["prediction"]
)

In [ ]:
precision_score(
    validation_woe["SeriousDlqin2yrs"],
    validation_woe["prediction"]
)

In [ ]:
recall_score(
    validation_woe["SeriousDlqin2yrs"],
    validation_woe["prediction"]
)

In [ ]:
f1_score(
    validation_woe["SeriousDlqin2yrs"],
    validation_woe["prediction"]
)

In [ ]:
specificity = tn / (tn + fp)

print(specificity)

In [ ]:
metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1"
    ],
    "Value": [
        accuracy_score(
            validation_woe["SeriousDlqin2yrs"],
            validation_woe["prediction"]
        ),

        precision_score(
            validation_woe["SeriousDlqin2yrs"],
            validation_woe["prediction"]
        ),

        recall_score(
            validation_woe["SeriousDlqin2yrs"],
            validation_woe["prediction"]
        ),

        specificity,

        f1_score(
            validation_woe["SeriousDlqin2yrs"],
            validation_woe["prediction"]
        )
    ]
})

metrics

## Confusion Matrix Interpretation

Using an illustrative PD cut-off of 10%, the model achieves:

- Accuracy: 83.5%
- Precision: 23.0%
- Recall: 62.6%
- Specificity: 85.0%
- F1-score: 33.6%

The model captures approximately 63% of bad customers while correctly identifying approximately 85% of good customers.

Precision is relatively low, which is expected in an imbalanced credit risk dataset with a low bad rate. In this context, recall and specificity are more informative than accuracy alone.

The 10% PD cut-off provides a relatively conservative risk policy, prioritising bad customer detection over maximum approval rate.

In [ ]:
train_woe.to_csv(
    "../data/processed/train_scoring.csv",
    index=False
)

validation_woe.to_csv(
    "../data/processed/validation_scoring.csv",
    index=False
)

test_woe.to_csv(
    "../data/processed/test_scoring.csv",
    index=False
)